In [ ]:
from TemporalHybridTransformer_EF import HybridTransformer as TMMF
from Metrics import log_metrics as metrics

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score
import pandas as pd
import os
import pickle

torch.manual_seed(42)
torch.cuda.manual_seed(42)

class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

		self.model = TMMF(
			config['depth'],
			config['query_size'],
			config['key_size'],
			config['value_size'],
			config['emb_size'],
			config['num_heads'],
			config['expansion'],
			config['conv_dropout'],
			config['self_dropout'],
			config['cross_dropout'],
			config['cls_dropout'],
			config['num_classes'],
			self.device,
		).to(self.device).to(torch.float64)

		self.optimizer = optim.Adam(self.model.parameters(), lr=config['learning_rate'])
		self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, 'min')
		self.criterion = nn.CrossEntropyLoss()

		os.makedirs('Results', exist_ok=True)
	
	def z_score(self, signal, eps=1e-6):
		mean = signal.mean(dim=(0,2), keepdim=True)
		std = signal.std(dim=(0,2), keepdim=True)
		std = torch.max(std, torch.tensor(eps, device=signal.device))
		signal = (signal - mean) / std
		return signal	
	
	def train_epoch(self, epoch, train_loader):
		epoch += 1
		self.model.train()
		total_correct, total_loss = 0, 0

		for batch_eeg, batch_nirs, batch_labels in train_loader:
			batch_eeg = batch_eeg.to(self.device, dtype=torch.float64)
			batch_nirs = batch_nirs.to(self.device, dtype=torch.float64)
			batch_labels = batch_labels.to(self.device)
			
			outputs = self.model(batch_eeg, batch_nirs)
			cls_loss = self.criterion(outputs, batch_labels)
			self.optimizer.zero_grad()
			cls_loss.backward()
			self.optimizer.step()
			
			preds = torch.argmax(outputs, dim=1)
			total_correct += (preds == batch_labels).sum().item()
			total_loss += cls_loss.item()

		train_loss = total_loss / len(train_loader)
		train_acc = total_correct / len(train_loader.dataset)
		
		return train_loss, train_acc
	
	def evaluate_epoch(self, eval_loader):
		self.model.eval()
		total_loss, total_correct = 0, 0
		all_preds, all_labels = [], []
		
		with torch.no_grad():
			for eval_eeg, eval_nirs, eval_labels in eval_loader:
				eval_eeg = eval_eeg.to(self.device, dtype=torch.float64)
				eval_nirs = eval_nirs.to(self.device, dtype=torch.float64)
				eval_labels = eval_labels.to(self.device)
				
				outputs = self.model(eval_eeg, eval_nirs)
				cls_loss = self.criterion(outputs, eval_labels)
				total_loss += cls_loss.item()
				
				preds = torch.argmax(outputs, dim=1)
				total_correct += (preds == eval_labels).sum().item()
				
				all_preds.extend(preds.cpu().numpy())
				all_labels.extend(eval_labels.cpu().numpy())
		
		loss = total_loss / len(eval_loader)
		acc = total_correct / len(eval_loader.dataset)
		precision = precision_score(all_labels, all_preds, zero_division=0)
		recall = recall_score(all_labels, all_preds)
		f1 = f1_score(all_labels, all_preds)
		kappa = cohen_kappa_score(all_labels, all_preds)
		
		return loss, acc, precision, recall, f1, kappa

	def train_subject(self, subject, mode):
		if mode == 0:
			data_root = config['mi_root'] + str(subject) + '.pkl'
		elif mode == 1:
			data_root = config['ma_root'] + str(subject) + '.pkl'
		elif mode == 2:
			data_root = config['wg_root'] + f'subject:{subject:02d}' + '.pkl'
		with open(data_root, 'rb') as f:
			data = pickle.load(f)
		
		eeg = data['eeg']
		nirs = data['nirs']
		labels = data['labels']

		# eeg = self.z_score(eeg)
		# nirs = self.z_score(nirs)
		
		# eeg = eeg.unsqueeze(1)
		# nirs = nirs.unsqueeze(1)
		eeg, nirs, labels = eeg.to(self.device), nirs.to(self.device), labels.to(self.device)
		
		train_size = int(config['ratio'] * len(eeg)) # training/testing ratio
		eval_size = len(eeg) - train_size
		
		dataset = torch.utils.data.TensorDataset(eeg, nirs, labels)
		train_dataset, eval_dataset = torch.utils.data.random_split(dataset, [train_size, eval_size])
		train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=self.config['batch_size'], shuffle=True)
		eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
		
		acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
		
		for epoch in range(self.config['num_epochs']):
			train_loss, train_acc = self.train_epoch(epoch, train_loader)
			eval_loss, eval_acc, precision, recall, f1, kappa = self.evaluate_epoch(eval_loader)
			
			acc_list.append(eval_acc)
			precision_list.append(precision)
			recall_list.append(recall)
			f1_list.append(f1)
			kappa_list.append(kappa)
			
			print(f"Subject {subject} | Epoch {epoch+1}/{self.config['num_epochs']} | "
				f"Train Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f} | "
				f"Eval Loss: {eval_loss:.2f} | Eval Acc: {eval_acc:.2f}")
			
		metrics(subject, config['log_name'], config['log_mode'], acc_list, precision_list, recall_list, f1_list, kappa_list)
		return

# Hyperparameters
# the correlation between Q, K, V and embedding size
config = {
	'depth': 4,
	'query_size': 64,
	'key_size': 64,
	'value_size': 64,
	'emb_size': 64,
	'decay': 0.99,
	'num_heads': 4,
	'expansion': 2,
	'conv_dropout': 0.3,
	'self_dropout': 0.3,
	'cross_dropout': 0.3,
	'cls_dropout': 0.5,
	'num_classes': 2,
	'batch_size': 16,
	'num_epochs': 200,
	'learning_rate': 1e-3,
	'ratio': 0.6,
	'log_name': '70',
	'log_mode': 1,
	'mi_root': '../../Dataset/EF-MI-MA/EF-PKL-MI/',
	'ma_root': '../../Dataset/EF-MI-MA/EF-PKL-MA/',
	'wg_root': '../../Dataset/EF-WG/WG/',
}

# Initialize and run trainer
trainer = Trainer(config)
for subject in range(29):
	subject += 1
	print(f"\n=== Subject {subject} ===")
	results = trainer.train_subject(subject, mode=1) # 0 = MI, 1 = MA, 2 = WG

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Start multi-head attention

=== Subject 1 ===
Subject 1 | Epoch 1/200 | Train Loss: 0.76 | Train Acc: 0.47 | Eval Loss: 1.29 | Eval Acc: 0.38
Subject 1 | Epoch 2/200 | Train Loss: 0.69 | Train Acc: 0.58 | Eval Loss: 0.78 | Eval Acc: 0.38
Subject 1 | Epoch 3/200 | Train Loss: 0.62 | Train Acc: 0.75 | Eval Loss: 0.65 | Eval Acc: 0.67
Subject 1 | Epoch 4/200 | Train Loss: 0.53 | Train Acc: 0.64 | Eval Loss: 0.85 | Eval Acc: 0.38
Subject 1 | Epoch 5/200 | Train Loss: 0.40 | Train Acc: 0.83 | Eval Loss: 0.94 | Eval Acc: 0.38
Subject 1 | Epoch 6/200 | Train Loss: 0.22 | Train Acc: 0.92 | Eval Loss: 0.54 | Eval Acc: 0.67
Subject 1 | Epoch 7/200 | Train Loss: 0.11 | Train Acc: 1.00 | Eval Loss: 0.44 | Eval Acc: 0.83
Subject 1 | Epoch 8/200 | Train Loss: 0.04 | Train Acc: 1.00 | Eval Loss: 0.58 | Eval Acc: 0.75
Subject 1 | Epoch 9/200 | Train Loss: 0.29 | Train Acc: 0.89 | Eval Loss: 0.70 | Eval Acc: 0.75
Subject 1 | Epoch 10/200 | Train Loss: 0.45 | Train Acc: 0.97 | Eval Loss: 0.88 | Eval Acc